# Módulo 1 — Capítulo 3: Análise e Transmissão de Sinais

**Curso de Princípios de Comunicação — UnB**

Este notebook apresenta exemplos práticos e interativos para os 9 tópicos do Capítulo 3. Cada seção é autocontida e pode ser executada independentemente (desde que a célula de configuração inicial seja executada primeiro).

---

**Sumário:**
1. [Seção 3.1 — Transformada de Fourier de Sinais](#sec3_1)
2. [Seção 3.2 — Transformadas de Funções Úteis](#sec3_2)
3. [Seção 3.3 — Propriedades da Transformada de Fourier](#sec3_3)
4. [Seção 3.4 — Transmissão através de Sistemas LTI](#sec3_4)
5. [Seção 3.5 — Filtros Ideais vs Práticos](#sec3_5)
6. [Seção 3.6 — Distorção de Sinais](#sec3_6)
7. [Seção 3.7 — Densidade Espectral de Energia](#sec3_7)
8. [Seção 3.8 — Densidade Espectral de Potência](#sec3_8)
9. [Seção 3.9 — Transformada Discreta de Fourier (DFT) e FFT](#sec3_9)

In [ ]:
# Configuração Inicial
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, fft
from numpy.fft import fft as npfft, ifft, fftfreq, fftshift

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'cm'

# Estilo limpo para os gráficos
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

print("Bibliotecas carregadas com sucesso!")

<a id="sec3_1"></a>
## Seção 3.1: Transformada de Fourier de Sinais

A Transformada de Fourier (TF) permite representar um sinal no domínio da frequência. Para um sinal $g(t)$ absolutamente integrável, a TF é definida como:

$$G(f) = \int_{-\infty}^{\infty} g(t) \, e^{-j2\pi f t} \, dt$$

e a transformada inversa:

$$g(t) = \int_{-\infty}^{\infty} G(f) \, e^{j2\pi f t} \, df$$

Vamos explorar a TF de dois sinais clássicos: o **pulso retangular** e a **exponencial decrescente**.

In [ ]:
# --- 3.1a: Transformada de Fourier do Pulso Retangular ---
# Pulso retangular de largura τ: g(t) = 1 para |t| < τ/2, 0 caso contrário
# TF analítica: G(f) = τ sinc(fτ)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

taus = [0.5, 1.0, 2.0]
t = np.linspace(-3, 3, 2000)
f = np.linspace(-8, 8, 2000)

for i, tau in enumerate(taus):
    # Domínio do tempo
    g_t = np.where(np.abs(t) <= tau / 2, 1.0, 0.0)
    axes[0, i].plot(t, g_t, 'b', linewidth=2)
    axes[0, i].set_title(f'Pulso retangular, $\\tau = {tau}$')
    axes[0, i].set_xlabel('$t$ (s)')
    axes[0, i].set_ylabel('$g(t)$')
    axes[0, i].set_ylim(-0.2, 1.4)
    axes[0, i].grid(True)

    # Domínio da frequência (analítico)
    G_f = tau * np.sinc(f * tau)  # np.sinc(x) = sin(πx)/(πx)
    axes[1, i].plot(f, G_f, 'r', linewidth=2)
    axes[1, i].set_title(f'$G(f) = \\tau \\, \\mathrm{{sinc}}(f\\tau)$, $\\tau = {tau}$')
    axes[1, i].set_xlabel('$f$ (Hz)')
    axes[1, i].set_ylabel('$G(f)$')
    axes[1, i].axhline(y=0, color='k', linewidth=0.5)
    axes[1, i].grid(True)

fig.suptitle('Relação tempo-largura de banda: pulso mais estreito → espectro mais largo',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.1b: Transformada de Fourier da Exponencial Decrescente ---
# g(t) = e^{-at} u(t), a > 0
# TF analítica: G(f) = 1 / (a + j2πf)
# |G(f)| = 1 / sqrt(a² + (2πf)²)
# ∠G(f) = -arctan(2πf / a)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

a_values = [1, 2, 5]
t = np.linspace(-1, 5, 2000)
f = np.linspace(-10, 10, 2000)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Domínio do tempo
for i, a in enumerate(a_values):
    g_t = np.exp(-a * t) * (t >= 0)
    axes[0, 0].plot(t, g_t, color=colors[i], linewidth=2, label=f'$a = {a}$')
axes[0, 0].set_title('Sinal: $g(t) = e^{-at} u(t)$')
axes[0, 0].set_xlabel('$t$ (s)')
axes[0, 0].set_ylabel('$g(t)$')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Magnitude do espectro
for i, a in enumerate(a_values):
    mag = 1.0 / np.sqrt(a**2 + (2 * np.pi * f)**2)
    axes[0, 1].plot(f, mag, color=colors[i], linewidth=2, label=f'$a = {a}$')
axes[0, 1].set_title('Espectro de magnitude: $|G(f)|$')
axes[0, 1].set_xlabel('$f$ (Hz)')
axes[0, 1].set_ylabel('$|G(f)|$')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Fase do espectro
for i, a in enumerate(a_values):
    phase = -np.arctan2(2 * np.pi * f, a)
    axes[1, 0].plot(f, np.degrees(phase), color=colors[i], linewidth=2, label=f'$a = {a}$')
axes[1, 0].set_title('Espectro de fase: $\\angle G(f)$')
axes[1, 0].set_xlabel('$f$ (Hz)')
axes[1, 0].set_ylabel('Fase (graus)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Magnitude em dB
for i, a in enumerate(a_values):
    mag = 1.0 / np.sqrt(a**2 + (2 * np.pi * f)**2)
    axes[1, 1].plot(f, 20 * np.log10(mag / np.max(mag)), color=colors[i], linewidth=2, label=f'$a = {a}$')
axes[1, 1].set_title('Espectro de magnitude (dB normalizado)')
axes[1, 1].set_xlabel('$f$ (Hz)')
axes[1, 1].set_ylabel('$|G(f)|$ (dB)')
axes[1, 1].set_ylim(-40, 5)
axes[1, 1].axhline(y=-3, color='gray', linestyle='--', label='-3 dB')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

**Observações da Seção 3.1:**

- **Relação tempo-largura de banda**: Quanto mais estreito o pulso no tempo ($\tau$ menor), mais largo é o seu espectro em frequência. Essa é uma manifestação do *princípio da incerteza*.
- Para a exponencial $e^{-at}u(t)$: valores maiores de $a$ significam decaimento mais rápido no tempo e espectro mais largo (largura de banda de 3 dB proporcional a $a$).
- A fase da exponencial é sempre negativa para $f > 0$, indicando atraso — característica de sistemas causais.

---

<a id="sec3_2"></a>
## Seção 3.2: Transformadas de Funções Úteis

Existem diversas funções que aparecem frequentemente em análise de sinais e comunicações. Nesta seção, visualizamos os pares de transformadas mais comuns:

| Função | $g(t)$ | $G(f)$ |
|--------|--------|--------|
| Delta de Dirac | $\delta(t)$ | $1$ |
| Retangular | $\mathrm{rect}(t/\tau)$ | $\tau \, \mathrm{sinc}(f\tau)$ |
| Triangular | $\mathrm{tri}(t/\tau)$ | $\tau \, \mathrm{sinc}^2(f\tau)$ |
| Gaussiana | $e^{-\pi t^2}$ | $e^{-\pi f^2}$ |
| Exponencial bilateral | $e^{-a|t|}$ | $\frac{2a}{a^2 + (2\pi f)^2}$ |

In [ ]:
# --- 3.2: Pares de Transformadas de Fourier de Funções Úteis ---

t = np.linspace(-3, 3, 4000)
f = np.linspace(-5, 5, 4000)

# Definição dos pares
funcoes = {
    'Delta (aprox.)': {
        'tempo': lambda t: np.exp(-500 * t**2) * np.sqrt(500 / np.pi),  # Gaussiana estreita
        'freq': lambda f: np.ones_like(f),
        'eq_t': r'$\delta(t) \approx$ Gaussiana estreita',
        'eq_f': r'$G(f) = 1$'
    },
    'Retangular': {
        'tempo': lambda t: np.where(np.abs(t) <= 0.5, 1.0, 0.0),
        'freq': lambda f: np.sinc(f),
        'eq_t': r'$\mathrm{rect}(t)$',
        'eq_f': r'$\mathrm{sinc}(f)$'
    },
    'Triangular': {
        'tempo': lambda t: np.maximum(1 - np.abs(t), 0),
        'freq': lambda f: np.sinc(f)**2,
        'eq_t': r'$\mathrm{tri}(t)$',
        'eq_f': r'$\mathrm{sinc}^2(f)$'
    },
    'Gaussiana': {
        'tempo': lambda t: np.exp(-np.pi * t**2),
        'freq': lambda f: np.exp(-np.pi * f**2),
        'eq_t': r'$e^{-\pi t^2}$',
        'eq_f': r'$e^{-\pi f^2}$'
    },
    'Exp. bilateral': {
        'tempo': lambda t: np.exp(-2 * np.abs(t)),
        'freq': lambda f: 4 / (4 + (2 * np.pi * f)**2),
        'eq_t': r'$e^{-2|t|}$',
        'eq_f': r'$\frac{4}{4 + (2\pi f)^2}$'
    }
}

fig, axes = plt.subplots(5, 2, figsize=(12, 16))

for i, (nome, dados) in enumerate(funcoes.items()):
    # Domínio do tempo
    axes[i, 0].plot(t, dados['tempo'](t), 'b', linewidth=2)
    axes[i, 0].set_title(f'{nome} — Tempo: {dados["eq_t"]}')
    axes[i, 0].set_xlabel('$t$')
    axes[i, 0].set_ylabel('$g(t)$')
    axes[i, 0].grid(True)

    # Domínio da frequência
    axes[i, 1].plot(f, dados['freq'](f), 'r', linewidth=2)
    axes[i, 1].set_title(f'{nome} — Frequência: {dados["eq_f"]}')
    axes[i, 1].set_xlabel('$f$ (Hz)')
    axes[i, 1].set_ylabel('$G(f)$')
    axes[i, 1].grid(True)

plt.tight_layout()
plt.show()

**Observações da Seção 3.2:**

- A **função delta** possui espectro constante (todas as frequências com mesma amplitude) — é a função mais "larga" possível em frequência.
- A **Gaussiana** é a única função que é *autotransformada* — sua TF também é uma Gaussiana. Ela minimiza o produto tempo-largura de banda.
- A **função triangular** é a convolução de dois pulsos retangulares, e portanto sua TF é $\mathrm{sinc}^2(f)$, que decai mais rapidamente que $\mathrm{sinc}(f)$.
- A **exponencial bilateral** tem espectro Lorentziano (sem zeros, decaimento suave).

---

<a id="sec3_3"></a>
## Seção 3.3: Propriedades da Transformada de Fourier

As propriedades da TF são ferramentas essenciais que permitem calcular transformadas de sinais complexos a partir de pares conhecidos. As principais propriedades são:

1. **Linearidade**: $a g_1(t) + b g_2(t) \leftrightarrow a G_1(f) + b G_2(f)$
2. **Deslocamento temporal**: $g(t - t_0) \leftrightarrow G(f) e^{-j2\pi f t_0}$
3. **Modulação (deslocamento em frequência)**: $g(t) \cos(2\pi f_0 t) \leftrightarrow \frac{1}{2}[G(f - f_0) + G(f + f_0)]$
4. **Escala**: $g(at) \leftrightarrow \frac{1}{|a|} G(f/a)$
5. **Convolução**: $g_1(t) * g_2(t) \leftrightarrow G_1(f) \cdot G_2(f)$
6. **Parseval**: $\int |g(t)|^2 dt = \int |G(f)|^2 df$

In [ ]:
# --- 3.3a: Linearidade ---
# TF(a·g1 + b·g2) = a·G1 + b·G2

N = 4096
dt = 0.01
t = np.arange(-N//2, N//2) * dt
freqs = fftfreq(N, dt)

# Dois sinais
g1 = np.exp(-np.abs(t))           # Exponencial bilateral
g2 = np.where(np.abs(t) <= 1, 1.0, 0.0)  # Pulso retangular

a, b = 2.0, 0.5
g_soma = a * g1 + b * g2

# FFT de cada sinal e da soma
G1 = fftshift(npfft(g1)) * dt
G2 = fftshift(npfft(g2)) * dt
G_soma = fftshift(npfft(g_soma)) * dt
G_linear = a * G1 + b * G2
f_plot = fftshift(freqs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t, g1, label='$g_1(t) = e^{-|t|}$')
axes[0].plot(t, g2, label=r'$g_2(t) = \mathrm{rect}(t/2)$')
axes[0].plot(t, g_soma, 'k--', linewidth=2, label=f'${a}g_1 + {b}g_2$')
axes[0].set_title('Domínio do tempo')
axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$g(t)$')
axes[0].legend(); axes[0].set_xlim(-5, 5); axes[0].grid(True)

axes[1].plot(f_plot, np.abs(G_soma), 'b', linewidth=2, label='FFT da soma')
axes[1].plot(f_plot, np.abs(G_linear), 'r--', linewidth=2, label='Soma das FFTs')
axes[1].set_title('$|G(f)|$: Verificação da linearidade')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('$|G(f)|$')
axes[1].legend(); axes[1].set_xlim(-5, 5); axes[1].grid(True)

erro = np.max(np.abs(G_soma - G_linear))
axes[2].plot(f_plot, np.abs(G_soma - G_linear), 'g', linewidth=2)
axes[2].set_title(f'Erro (máx = {erro:.2e})')
axes[2].set_xlabel('$f$ (Hz)'); axes[2].set_ylabel('Erro absoluto')
axes[2].set_xlim(-5, 5); axes[2].grid(True)

plt.tight_layout()
plt.show()
print(f"Erro máximo entre FFT(soma) e soma(FFTs): {erro:.2e} → Linearidade verificada!")

In [ ]:
# --- 3.3b: Deslocamento Temporal ---
# g(t - t0) ↔ G(f)·e^{-j2πft0}  →  magnitude não muda, fase muda linearmente

N = 4096
dt = 0.01
t = np.arange(-N//2, N//2) * dt
freqs = fftshift(fftfreq(N, dt))

# Sinal original: Gaussiana
g_orig = np.exp(-2 * t**2)
t0 = 3.0  # deslocamento
g_shift = np.exp(-2 * (t - t0)**2)

G_orig = fftshift(npfft(g_orig)) * dt
G_shift = fftshift(npfft(g_shift)) * dt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t, g_orig, 'b', linewidth=2, label='$g(t)$')
axes[0].plot(t, g_shift, 'r--', linewidth=2, label=f'$g(t - {t0})$')
axes[0].set_title('Domínio do tempo')
axes[0].set_xlabel('$t$'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(freqs, np.abs(G_orig), 'b', linewidth=2, label='$|G_{orig}(f)|$')
axes[1].plot(freqs, np.abs(G_shift), 'r--', linewidth=2, label='$|G_{shift}(f)|$')
axes[1].set_title('Magnitude (inalterada)')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_xlim(-3, 3); axes[1].legend(); axes[1].grid(True)

axes[2].plot(freqs, np.angle(G_orig), 'b', linewidth=2, label='Fase original')
axes[2].plot(freqs, np.angle(G_shift), 'r--', linewidth=2, label='Fase deslocada')
axes[2].set_title('Fase (alterada linearmente)')
axes[2].set_xlabel('$f$ (Hz)'); axes[2].set_xlim(-3, 3); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()
print(f"Deslocamento de t₀ = {t0} s adiciona fase linear -2πf·t₀ ao espectro.")

In [ ]:
# --- 3.3c: Modulação ---
# g(t)·cos(2πf₀t) ↔ ½[G(f-f₀) + G(f+f₀)]

N = 8192
dt = 0.005
t = np.arange(-N//2, N//2) * dt
freqs = fftshift(fftfreq(N, dt))

# Sinal banda-base (Gaussiana)
g = np.exp(-2 * t**2)
f0 = 5.0  # frequência da portadora
g_mod = g * np.cos(2 * np.pi * f0 * t)

G = fftshift(npfft(g)) * dt
G_mod = fftshift(npfft(g_mod)) * dt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(t, g, 'b', linewidth=2)
axes[0, 0].set_title('Sinal original $g(t)$')
axes[0, 0].set_xlabel('$t$'); axes[0, 0].set_xlim(-3, 3); axes[0, 0].grid(True)

axes[0, 1].plot(t, g_mod, 'r', linewidth=1.5)
axes[0, 1].set_title(f'Sinal modulado $g(t)\\cos(2\\pi \\cdot {f0} \\cdot t)$')
axes[0, 1].set_xlabel('$t$'); axes[0, 1].set_xlim(-3, 3); axes[0, 1].grid(True)

axes[1, 0].plot(freqs, np.abs(G), 'b', linewidth=2)
axes[1, 0].set_title('Espectro original $|G(f)|$')
axes[1, 0].set_xlabel('$f$ (Hz)'); axes[1, 0].set_xlim(-15, 15); axes[1, 0].grid(True)

axes[1, 1].plot(freqs, np.abs(G_mod), 'r', linewidth=2)
axes[1, 1].axvline(x=f0, color='gray', linestyle='--', alpha=0.7, label=f'$f_0 = {f0}$ Hz')
axes[1, 1].axvline(x=-f0, color='gray', linestyle='--', alpha=0.7)
axes[1, 1].set_title('Espectro modulado: deslocamento para $\\pm f_0$')
axes[1, 1].set_xlabel('$f$ (Hz)'); axes[1, 1].set_xlim(-15, 15)
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.show()
print("A modulação desloca o espectro para ±f₀, dividindo a amplitude por 2.")

In [ ]:
# --- 3.3d: Escala ---
# g(at) ↔ (1/|a|) G(f/a)
# Compressão no tempo → expansão na frequência e vice-versa

N = 4096
dt = 0.01
t = np.arange(-N//2, N//2) * dt
freqs = fftshift(fftfreq(N, dt))

# Gaussiana com diferentes escalas
escalas = [0.5, 1.0, 2.0]
colors = ['#e74c3c', '#3498db', '#2ecc71']
labels_t = [r'$g(2t)$ (comprimido)', r'$g(t)$ (original)', r'$g(t/2)$ (expandido)']
labels_f = [r'$\frac{1}{2}G(f/2)$', r'$G(f)$', r'$2G(2f)$']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, a in enumerate(escalas):
    g = np.exp(-np.pi * (a * t)**2)
    G = fftshift(npfft(g)) * dt

    axes[0].plot(t, g, color=colors[i], linewidth=2, label=labels_t[i])
    axes[1].plot(freqs, np.abs(G), color=colors[i], linewidth=2, label=labels_f[i])

axes[0].set_title('Domínio do tempo: escala temporal')
axes[0].set_xlabel('$t$'); axes[0].set_xlim(-4, 4)
axes[0].legend(); axes[0].grid(True)

axes[1].set_title('Domínio da frequência: efeito recíproco')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_xlim(-3, 3)
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()
print("Compressão no tempo por fator a → expansão no espectro por fator a (e vice-versa).")

In [ ]:
# --- 3.3e: Convolução ---
# g1(t) * g2(t) ↔ G1(f) · G2(f)

N = 4096
dt = 0.01
t = np.arange(-N//2, N//2) * dt
freqs = fftshift(fftfreq(N, dt))

# Dois sinais
g1 = np.where(np.abs(t) <= 1, 1.0, 0.0)    # Pulso retangular
g2 = np.exp(-2 * np.abs(t))                  # Exponencial bilateral

# Convolução no tempo (via FFT)
G1 = npfft(g1)
G2 = npfft(g2)
g_conv = np.real(ifft(G1 * G2)) * dt  # convolução via multiplicação espectral
G_conv = fftshift(npfft(g_conv)) * dt

# Para comparação direta
G1_plot = fftshift(G1) * dt
G2_plot = fftshift(G2) * dt
G_prod = G1_plot * G2_plot / dt  # produto no domínio da frequência

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(t, g1, 'b', linewidth=2, label='$g_1(t)$ (rect)')
axes[0, 0].plot(t, g2, 'r', linewidth=2, label='$g_2(t)$ (exp)')
axes[0, 0].set_title('Sinais originais')
axes[0, 0].set_xlabel('$t$'); axes[0, 0].set_xlim(-5, 5)
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(t, g_conv, 'purple', linewidth=2)
axes[0, 1].set_title('Convolução $g_1 * g_2$ (no tempo)')
axes[0, 1].set_xlabel('$t$'); axes[0, 1].set_xlim(-5, 5); axes[0, 1].grid(True)

axes[1, 0].plot(freqs, np.abs(G1_plot), 'b', linewidth=2, label='$|G_1(f)|$')
axes[1, 0].plot(freqs, np.abs(G2_plot), 'r', linewidth=2, label='$|G_2(f)|$')
axes[1, 0].set_title('Espectros individuais')
axes[1, 0].set_xlabel('$f$ (Hz)'); axes[1, 0].set_xlim(-5, 5)
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(freqs, np.abs(G_conv), 'purple', linewidth=2, label='FFT da convolução')
axes[1, 1].plot(freqs, np.abs(G_prod), 'k--', linewidth=2, label='$G_1 \\cdot G_2$')
axes[1, 1].set_title('Verificação: FFT(conv) = produto')
axes[1, 1].set_xlabel('$f$ (Hz)'); axes[1, 1].set_xlim(-5, 5)
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.show()
print("Convolução no tempo ↔ multiplicação na frequência: verificado!")

In [ ]:
# --- 3.3f: Teorema de Parseval ---
# ∫|g(t)|² dt = ∫|G(f)|² df  (energia nos dois domínios é igual)

N = 4096
dt = 0.01
t = np.arange(-N//2, N//2) * dt
df = 1 / (N * dt)

# Vários sinais para verificar
sinais = {
    'Gaussiana': np.exp(-np.pi * t**2),
    'Exponencial': np.exp(-2 * np.abs(t)),
    'Pulso retangular': np.where(np.abs(t) <= 1, 1.0, 0.0),
    'Sinc': np.sinc(t),
}

print("Verificação do Teorema de Parseval:")
print(f"{'Sinal':<25} {'E_tempo':>12} {'E_freq':>12} {'Erro relativo':>15}")
print("-" * 68)

for nome, g in sinais.items():
    E_tempo = np.sum(np.abs(g)**2) * dt
    G = fftshift(npfft(g)) * dt
    E_freq = np.sum(np.abs(G)**2) * df
    erro_rel = np.abs(E_tempo - E_freq) / E_tempo
    print(f"{nome:<25} {E_tempo:>12.6f} {E_freq:>12.6f} {erro_rel:>15.2e}")

print("\nResultado: a energia calculada no tempo e na frequência é a mesma (erros numéricos desprezíveis).")

**Resumo das Propriedades (Seção 3.3):**

- **Linearidade**: a TF é uma operação linear — fundamental para análise de sistemas.
- **Deslocamento temporal**: atraso no tempo equivale a rotação de fase, sem alterar a magnitude.
- **Modulação**: base da transmissão de sinais em comunicações — desloca o espectro para a frequência da portadora.
- **Escala**: princípio da incerteza em ação — impossível ter sinal estreito no tempo E na frequência simultaneamente.
- **Convolução**: permite calcular a saída de um sistema LTI via multiplicação na frequência.
- **Parseval**: a energia total é preservada entre os domínios — base para análise espectral de energia.

---

<a id="sec3_4"></a>
## Seção 3.4: Transmissão através de Sistemas LTI

Um sistema Linear e Invariante no Tempo (LTI) é completamente caracterizado por sua **resposta ao impulso** $h(t)$ ou, equivalentemente, por sua **função de transferência** $H(f)$.

A saída $y(t)$ para uma entrada $x(t)$ é:

$$y(t) = x(t) * h(t) \quad \longleftrightarrow \quad Y(f) = X(f) \cdot H(f)$$

Vamos analisar o filtro RC passa-baixas clássico:

$$H(f) = \frac{1}{1 + j2\pi f RC}$$

com frequência de corte $f_c = \frac{1}{2\pi RC}$.

In [ ]:
# --- 3.4a: Filtro RC passa-baixas — Resposta em frequência ---

f = np.linspace(-50, 50, 10000)
RC_values = [0.01, 0.005, 0.002]  # Diferentes constantes RC
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, RC in enumerate(RC_values):
    fc = 1 / (2 * np.pi * RC)
    H = 1 / (1 + 1j * 2 * np.pi * f * RC)

    axes[0].plot(f, 20 * np.log10(np.abs(H)), color=colors[i], linewidth=2,
                 label=f'RC = {RC}, $f_c$ = {fc:.1f} Hz')
    axes[1].plot(f, np.degrees(np.angle(H)), color=colors[i], linewidth=2,
                 label=f'RC = {RC}')

axes[0].set_title('Resposta em magnitude do filtro RC')
axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_ylabel('$|H(f)|$ (dB)')
axes[0].axhline(y=-3, color='gray', linestyle='--', alpha=0.7, label='-3 dB')
axes[0].set_ylim(-40, 5); axes[0].legend(); axes[0].grid(True)

axes[1].set_title('Resposta em fase do filtro RC')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('Fase (graus)')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- 3.4b: Passando uma onda quadrada pelo filtro RC ---

N = 8192
fs = 1000  # taxa de amostragem
dt = 1 / fs
t = np.arange(N) * dt

# Onda quadrada (fundamental em 10 Hz)
f_square = 10
x = signal.square(2 * np.pi * f_square * t)

# Filtro RC com diferentes larguras de banda
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
RC_values = [0.002, 0.005, 0.02]  # fc ≈ 80, 32, 8 Hz

for i, RC in enumerate(RC_values):
    fc = 1 / (2 * np.pi * RC)

    # Filtrar no domínio da frequência
    freqs = fftfreq(N, dt)
    X = npfft(x)
    H = 1 / (1 + 1j * 2 * np.pi * freqs * RC)
    Y = X * H
    y = np.real(ifft(Y))

    # Gráfico no tempo
    axes[i, 0].plot(t, x, 'b', alpha=0.5, linewidth=1, label='Entrada (onda quadrada)')
    axes[i, 0].plot(t, y, 'r', linewidth=2, label='Saída filtrada')
    axes[i, 0].set_title(f'RC = {RC} s, $f_c$ = {fc:.1f} Hz')
    axes[i, 0].set_xlabel('$t$ (s)'); axes[i, 0].set_xlim(0, 0.3)
    axes[i, 0].legend(); axes[i, 0].grid(True)

    # Espectro
    f_plot = fftshift(freqs)
    axes[i, 1].plot(f_plot, np.abs(fftshift(X)) / N, 'b', alpha=0.5, label='$|X(f)|$')
    axes[i, 1].plot(f_plot, np.abs(fftshift(Y)) / N, 'r', linewidth=2, label='$|Y(f)|$')
    axes[i, 1].axvline(x=fc, color='gray', linestyle='--', alpha=0.7, label=f'$f_c = {fc:.0f}$ Hz')
    axes[i, 1].set_title(f'Espectro — $f_c$ = {fc:.1f} Hz')
    axes[i, 1].set_xlabel('$f$ (Hz)'); axes[i, 1].set_xlim(0, 200)
    axes[i, 1].legend(); axes[i, 1].grid(True)

plt.tight_layout()
plt.show()
print("Quanto menor a largura de banda (maior RC), mais suavizada fica a saída.")

**Observações da Seção 3.4:**

- O filtro RC é o exemplo mais simples de sistema LTI: atenua progressivamente as frequências acima de $f_c$.
- Quando $f_c$ é alta em relação à fundamental da onda quadrada, a saída preserva bem a forma. Quando $f_c$ é baixa, a filtragem remove as harmônicas e o sinal se torna quase senoidal.
- A multiplicação no domínio da frequência ($Y = X \cdot H$) é equivalente à convolução no tempo — é a essência da análise de sistemas LTI.

---

<a id="sec3_5"></a>
## Seção 3.5: Filtros Ideais vs Práticos

O **filtro passa-baixas ideal** tem resposta em magnitude retangular:

$$H_{ideal}(f) = \begin{cases} 1, & |f| \leq f_c \\ 0, & |f| > f_c \end{cases}$$

Na prática, filtros ideais são **não-causais** (resposta ao impulso sinc não é causal). Utilizamos aproximações como:

- **Butterworth**: máxima planura na banda passante
- **Chebyshev Tipo I**: ripple na banda passante, transição mais acentuada
- **Bessel**: fase mais linear (menor distorção de grupo)

In [ ]:
# --- 3.5a: Butterworth de diferentes ordens vs filtro ideal ---

fc = 100  # Hz, frequência de corte
fs = 1000  # taxa de amostragem
f = np.linspace(0, fs/2, 5000)
w = 2 * np.pi * f / fs  # frequência normalizada para Bode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filtro ideal
H_ideal = np.where(f <= fc, 1.0, 0.0)
axes[0].plot(f, 20 * np.log10(np.maximum(H_ideal, 1e-10)), 'k--', linewidth=2, label='Ideal')

# Butterworth de ordens 1, 2, 4, 8
orders = [1, 2, 4, 8]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for N_ord, cor in zip(orders, colors):
    b, a = signal.butter(N_ord, fc, btype='low', fs=fs)
    w_resp, h = signal.freqz(b, a, worN=f, fs=fs)
    axes[0].plot(f, 20 * np.log10(np.maximum(np.abs(h), 1e-10)), color=cor, linewidth=2,
                 label=f'Butterworth N={N_ord}')
    axes[1].plot(f, np.unwrap(np.angle(h)) * 180 / np.pi, color=cor, linewidth=2,
                 label=f'N={N_ord}')

axes[0].set_title('Magnitude: Filtro ideal vs Butterworth')
axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_ylabel('$|H(f)|$ (dB)')
axes[0].set_ylim(-80, 5); axes[0].axvline(x=fc, color='gray', linestyle=':', alpha=0.5)
axes[0].legend(); axes[0].grid(True)

axes[1].set_title('Fase: Butterworth de diferentes ordens')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('Fase (graus)')
axes[1].axvline(x=fc, color='gray', linestyle=':', alpha=0.5)
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()
print("Maior ordem → transição mais acentuada, aproximando-se do filtro ideal.")

In [ ]:
# --- 3.5b: Comparação Butterworth vs Chebyshev vs Bessel ---

fc = 100  # Hz
fs = 1000
N_ord = 4
f = np.linspace(0, fs/2, 5000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Butterworth
b_bw, a_bw = signal.butter(N_ord, fc, fs=fs)
_, h_bw = signal.freqz(b_bw, a_bw, worN=f, fs=fs)

# Chebyshev Tipo I (1 dB de ripple)
b_ch, a_ch = signal.cheby1(N_ord, 1, fc, fs=fs)
_, h_ch = signal.freqz(b_ch, a_ch, worN=f, fs=fs)

# Bessel
b_be, a_be = signal.bessel(N_ord, fc, btype='low', fs=fs, norm='mag')
_, h_be = signal.freqz(b_be, a_be, worN=f, fs=fs)

# Magnitude
axes[0].plot(f, 20*np.log10(np.maximum(np.abs(h_bw), 1e-10)), 'b', linewidth=2, label='Butterworth')
axes[0].plot(f, 20*np.log10(np.maximum(np.abs(h_ch), 1e-10)), 'r', linewidth=2, label='Chebyshev I (1 dB)')
axes[0].plot(f, 20*np.log10(np.maximum(np.abs(h_be), 1e-10)), 'g', linewidth=2, label='Bessel')
axes[0].set_title(f'Comparação de magnitude (ordem {N_ord})')
axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_ylabel('$|H(f)|$ (dB)')
axes[0].set_ylim(-60, 5); axes[0].axvline(x=fc, color='gray', linestyle=':', alpha=0.5)
axes[0].legend(); axes[0].grid(True)

# Atraso de grupo (group delay)
_, gd_bw = signal.group_delay((b_bw, a_bw), w=f, fs=fs)
_, gd_ch = signal.group_delay((b_ch, a_ch), w=f, fs=fs)
_, gd_be = signal.group_delay((b_be, a_be), w=f, fs=fs)

axes[1].plot(f, gd_bw / fs * 1000, 'b', linewidth=2, label='Butterworth')
axes[1].plot(f, gd_ch / fs * 1000, 'r', linewidth=2, label='Chebyshev I')
axes[1].plot(f, gd_be / fs * 1000, 'g', linewidth=2, label='Bessel')
axes[1].set_title('Atraso de grupo (group delay)')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('Atraso (ms)')
axes[1].set_xlim(0, 200); axes[1].axvline(x=fc, color='gray', linestyle=':', alpha=0.5)
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()
print("Chebyshev: transição mais acentuada, mas ripple na banda passante e atraso de grupo variável.")
print("Bessel: atraso de grupo mais constante (fase mais linear) — ideal para sinais pulsados.")

In [ ]:
# --- 3.5c: Aplicação prática — projetar e aplicar um filtro Butterworth ---

# Sinal com componentes em 20 Hz e 150 Hz
fs = 1000
N = 4096
t = np.arange(N) / fs

x = np.sin(2 * np.pi * 20 * t) + 0.5 * np.sin(2 * np.pi * 150 * t)

# Projetar filtro Butterworth passa-baixas de ordem 6, fc = 50 Hz
fc = 50
b, a = signal.butter(6, fc, fs=fs)

# Aplicar filtro (usando filtfilt para fase zero)
y = signal.filtfilt(b, a, x)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(t, x, 'b', alpha=0.7, label='Entrada: 20 Hz + 150 Hz')
axes[0].plot(t, y, 'r', linewidth=2, label='Saída filtrada')
axes[0].set_title('Filtragem no domínio do tempo')
axes[0].set_xlabel('$t$ (s)'); axes[0].set_xlim(0, 0.3)
axes[0].legend(); axes[0].grid(True)

# Espectro
freqs = fftshift(fftfreq(N, 1/fs))
X_fft = fftshift(npfft(x)) / N
Y_fft = fftshift(npfft(y)) / N

axes[1].plot(freqs, np.abs(X_fft), 'b', alpha=0.7, label='$|X(f)|$')
axes[1].plot(freqs, np.abs(Y_fft), 'r', linewidth=2, label='$|Y(f)|$')
axes[1].axvline(x=fc, color='gray', linestyle='--', label=f'$f_c = {fc}$ Hz')
axes[1].axvline(x=-fc, color='gray', linestyle='--')
axes[1].set_title('Espectro: entrada vs saída')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_xlim(-200, 200)
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f"O filtro Butterworth de ordem 6 com fc = {fc} Hz elimina a componente de 150 Hz,"
      f"\npreservando a componente de 20 Hz.")

**Observações da Seção 3.5:**

- Filtros de **ordem mais alta** possuem transição mais acentuada, mas introduzem mais atraso e podem causar instabilidade numérica.
- **Butterworth**: compromisso entre planura na banda passante e velocidade de transição.
- **Chebyshev**: transição mais rápida que Butterworth para mesma ordem, mas com ripple na banda passante.
- **Bessel**: atraso de grupo quase constante — preserva a forma de onda de pulsos.
- Na prática, `signal.filtfilt` aplica o filtro duas vezes (ida e volta), resultando em fase zero.

---

<a id="sec3_6"></a>
## Seção 3.6: Distorção de Sinais

Um sistema LTI causa **transmissão sem distorção** apenas se:

$$H(f) = K \, e^{-j2\pi f t_d}$$

ou seja, magnitude constante ($K$) e fase linear (atraso puro $t_d$).

Qualquer desvio dessas condições resulta em distorção:
- **Distorção de amplitude**: $|H(f)|$ não é constante na banda do sinal
- **Distorção de fase** (ou de atraso): atraso de grupo não é constante

Vamos demonstrar ambos os tipos e uma técnica de **equalização**.

In [ ]:
# --- 3.6a: Distorção de amplitude e de fase ---

N = 8192
fs = 1000
dt = 1 / fs
t = np.arange(N) * dt
freqs = fftfreq(N, dt)

# Sinal multi-tom (soma de senóides)
f_tons = [10, 30, 50, 70, 90]
x = sum(np.sin(2 * np.pi * f_k * t) for f_k in f_tons)

X = npfft(x)

# --- Canal com distorção de amplitude ---
# Magnitude que varia com a frequência (decaimento exponencial)
H_amp = np.exp(-np.abs(freqs) / 80)  # magnitude não-plana
Y_amp = X * H_amp
y_amp = np.real(ifft(Y_amp))

# --- Canal com distorção de fase ---
# Fase não-linear (quadrática)
fase_nl = -0.001 * (2 * np.pi * freqs)**2
H_fase = np.exp(1j * fase_nl)  # magnitude plana, fase não-linear
Y_fase = X * H_fase
y_fase = np.real(ifft(Y_fase))

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Sinal original
axes[0, 0].plot(t, x, 'b', linewidth=1)
axes[0, 0].set_title('Sinal original (5 tons)')
axes[0, 0].set_xlabel('$t$ (s)'); axes[0, 0].set_xlim(0, 0.2); axes[0, 0].grid(True)

axes[0, 1].stem(f_tons, [1]*5, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0, 1].set_title('Espectro do sinal original')
axes[0, 1].set_xlabel('$f$ (Hz)'); axes[0, 1].set_ylabel('Amplitude'); axes[0, 1].grid(True)

# Distorção de amplitude
axes[1, 0].plot(t, y_amp, 'r', linewidth=1)
axes[1, 0].set_title('Saída com distorção de amplitude')
axes[1, 0].set_xlabel('$t$ (s)'); axes[1, 0].set_xlim(0, 0.2); axes[1, 0].grid(True)

Y_amp_plot = np.abs(fftshift(Y_amp)) / N
f_plot = fftshift(freqs)
axes[1, 1].plot(f_plot, Y_amp_plot, 'r', linewidth=1)
axes[1, 1].set_title('Espectro: amplitudes diferentes para cada tom')
axes[1, 1].set_xlabel('$f$ (Hz)'); axes[1, 1].set_xlim(0, 120); axes[1, 1].grid(True)

# Distorção de fase
axes[2, 0].plot(t, y_fase, 'g', linewidth=1)
axes[2, 0].set_title('Saída com distorção de fase')
axes[2, 0].set_xlabel('$t$ (s)'); axes[2, 0].set_xlim(0, 0.2); axes[2, 0].grid(True)

Y_fase_plot = np.abs(fftshift(Y_fase)) / N
axes[2, 1].plot(f_plot, Y_fase_plot, 'g', linewidth=1)
axes[2, 1].set_title('Espectro: amplitudes iguais, mas fase alterada')
axes[2, 1].set_xlabel('$f$ (Hz)'); axes[2, 1].set_xlim(0, 120); axes[2, 1].grid(True)

plt.tight_layout()
plt.show()
print("Distorção de amplitude: altera as amplitudes relativas dos tons.")
print("Distorção de fase: mantém amplitudes mas altera o alinhamento temporal dos tons.")

In [ ]:
# --- 3.6b: Equalização — recuperação do sinal distorcido ---

N = 8192
fs = 1000
dt = 1 / fs
t = np.arange(N) * dt
freqs = fftfreq(N, dt)

# Sinal original
f_tons = [10, 30, 50, 70, 90]
x = sum(np.sin(2 * np.pi * f_k * t) for f_k in f_tons)
X = npfft(x)

# Canal distorcivo (amplitude + fase)
H_canal = np.exp(-np.abs(freqs) / 80) * np.exp(-1j * 0.0005 * (2 * np.pi * freqs)**2)
Y = X * H_canal
y = np.real(ifft(Y))

# Equalizador: H_eq = 1/H_canal (inverso do canal)
# Adicionar regularização para evitar divisão por zero
epsilon = 1e-6
H_eq = 1 / (H_canal + epsilon)
Y_eq = Y * npfft(np.real(ifft(npfft(np.ones(N)) * H_eq)))  # Simplificação:
# Na verdade, a equalização é: X_recuperado = Y · H_eq
X_rec = Y * H_eq
x_rec = np.real(ifft(X_rec))

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

axes[0].plot(t, x, 'b', linewidth=1.5)
axes[0].set_title('Sinal original $x(t)$')
axes[0].set_xlabel('$t$ (s)'); axes[0].set_xlim(0.1, 0.3); axes[0].grid(True)

axes[1].plot(t, y, 'r', linewidth=1.5)
axes[1].set_title('Sinal distorcido $y(t) = x(t) * h_c(t)$')
axes[1].set_xlabel('$t$ (s)'); axes[1].set_xlim(0.1, 0.3); axes[1].grid(True)

axes[2].plot(t, x, 'b', alpha=0.4, linewidth=1, label='Original')
axes[2].plot(t, x_rec, 'g', linewidth=1.5, label='Equalizado')
axes[2].set_title('Sinal recuperado após equalização: $\\hat{x}(t)$')
axes[2].set_xlabel('$t$ (s)'); axes[2].set_xlim(0.1, 0.3)
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()

erro_rms = np.sqrt(np.mean((x - x_rec)**2)) / np.sqrt(np.mean(x**2))
print(f"Erro RMS normalizado após equalização: {erro_rms:.4e}")
print("A equalização H_eq = 1/H_canal recupera o sinal original (dentro da precisão numérica).")

**Observações da Seção 3.6:**

- A **distorção de amplitude** altera as amplitudes relativas das componentes espectrais — muda a "forma" do sinal.
- A **distorção de fase** (atraso de grupo não-constante) dispersa os componentes no tempo — especialmente crítica para sinais pulsados.
- A **equalização** ($H_{eq} = 1/H_c$) compensa perfeitamente o canal quando não há ruído. Na prática, o ruído limita a eficácia da equalização.

---

<a id="sec3_7"></a>
## Seção 3.7: Densidade Espectral de Energia

A **Densidade Espectral de Energia** (DEE) de um sinal de energia $g(t)$ é definida como:

$$\Psi_g(f) = |G(f)|^2$$

A energia total do sinal pode ser calculada integrando a DEE:

$$E_g = \int_{-\infty}^{\infty} |g(t)|^2 \, dt = \int_{-\infty}^{\infty} |G(f)|^2 \, df = \int_{-\infty}^{\infty} \Psi_g(f) \, df$$

Conceitos importantes:
- **Largura de banda essencial**: faixa de frequências que contém uma fração significativa da energia (90%, 95%, 99%)
- **Princípio da incerteza**: $\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$

In [ ]:
# --- 3.7a: DEE e verificação de Parseval para diferentes sinais ---

N = 8192
dt = 0.005
t = np.arange(-N//2, N//2) * dt
df = 1 / (N * dt)
freqs = fftshift(fftfreq(N, dt))

sinais = {
    'Gaussiana': np.exp(-np.pi * t**2),
    'Pulso retangular': np.where(np.abs(t) <= 1, 1.0, 0.0),
    'Exponencial': np.exp(-2 * np.abs(t)),
}

fig, axes = plt.subplots(len(sinais), 2, figsize=(14, 4 * len(sinais)))

for i, (nome, g) in enumerate(sinais.items()):
    G = fftshift(npfft(g)) * dt
    ESD = np.abs(G)**2  # Densidade espectral de energia

    # Energia no tempo e na frequência
    E_t = np.sum(np.abs(g)**2) * dt
    E_f = np.sum(ESD) * df

    axes[i, 0].plot(t, np.abs(g)**2, 'b', linewidth=2)
    axes[i, 0].fill_between(t, np.abs(g)**2, alpha=0.3)
    axes[i, 0].set_title(f'{nome}: $|g(t)|^2$, $E_t = {E_t:.4f}$')
    axes[i, 0].set_xlabel('$t$'); axes[i, 0].set_xlim(-5, 5); axes[i, 0].grid(True)

    axes[i, 1].plot(freqs, ESD, 'r', linewidth=2)
    axes[i, 1].fill_between(freqs, ESD, alpha=0.3, color='red')
    axes[i, 1].set_title(f'{nome}: $\\Psi_g(f) = |G(f)|^2$, $E_f = {E_f:.4f}$')
    axes[i, 1].set_xlabel('$f$ (Hz)'); axes[i, 1].set_xlim(-5, 5); axes[i, 1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- 3.7b: Largura de banda essencial (contendo X% da energia) ---

N = 8192
dt = 0.005
t = np.arange(-N//2, N//2) * dt
df = 1 / (N * dt)
freqs = fftshift(fftfreq(N, dt))

# Pulso Gaussiano
g = np.exp(-np.pi * t**2)
G = fftshift(npfft(g)) * dt
ESD = np.abs(G)**2
E_total = np.sum(ESD) * df

# Calcular energia acumulada (apenas frequências positivas, por simetria)
f_pos = freqs[freqs >= 0]
ESD_pos = ESD[freqs >= 0]
E_acum = np.cumsum(ESD_pos) * df * 2  # ×2 por simetria
E_acum_norm = E_acum / E_total

# Encontrar larguras de banda para 90%, 95%, 99% da energia
percentuais = [0.90, 0.95, 0.99]
bws = []
for p in percentuais:
    idx = np.argmax(E_acum_norm >= p)
    bws.append(f_pos[idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(freqs, ESD, 'b', linewidth=2)
axes[0].set_title('Densidade Espectral de Energia $\\Psi_g(f)$')
axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_xlim(-3, 3); axes[0].grid(True)

# Sombrear regiões de largura de banda
colors_bw = ['#e74c3c', '#f39c12', '#2ecc71']
for bw, p, cor in zip(bws, percentuais, colors_bw):
    mask = np.abs(freqs) <= bw
    axes[0].fill_between(freqs[mask], ESD[mask], alpha=0.2, color=cor,
                         label=f'{int(p*100)}%: BW = {bw:.2f} Hz')
axes[0].legend()

axes[1].plot(f_pos, E_acum_norm * 100, 'b', linewidth=2)
axes[1].set_title('Energia acumulada (%)')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('% da energia total')
axes[1].set_xlim(0, 3); axes[1].grid(True)

for bw, p, cor in zip(bws, percentuais, colors_bw):
    axes[1].axhline(y=p*100, color=cor, linestyle='--', alpha=0.7)
    axes[1].axvline(x=bw, color=cor, linestyle='--', alpha=0.7)
    axes[1].plot(bw, p*100, 'o', color=cor, markersize=8)

plt.tight_layout()
plt.show()

print("Larguras de banda essenciais da Gaussiana:")
for bw, p in zip(bws, percentuais):
    print(f"  {int(p*100)}% da energia: BW = {bw:.3f} Hz")

In [ ]:
# --- 3.7c: Princípio da Incerteza — Δt·Δf para Gaussiana vs Retangular ---
# Δt = desvio padrão no tempo, Δf = desvio padrão na frequência
# Para Gaussiana: Δt·Δf = 1/(4π) (valor mínimo)

N = 16384
dt = 0.002
t = np.arange(-N//2, N//2) * dt
df = 1 / (N * dt)
freqs = fftshift(fftfreq(N, dt))

def calcular_delta_t_f(g, t, freqs, dt, df):
    """Calcula as durações RMS no tempo e frequência."""
    # Normalizar energia
    E = np.sum(np.abs(g)**2) * dt
    g_norm2 = np.abs(g)**2 / E

    # Duração RMS no tempo
    t_med = np.sum(t * g_norm2) * dt
    delta_t = np.sqrt(np.sum((t - t_med)**2 * g_norm2) * dt)

    # Espectro
    G = fftshift(npfft(g)) * dt
    G_norm2 = np.abs(G)**2 / (np.sum(np.abs(G)**2) * df)

    f_med = np.sum(freqs * G_norm2) * df
    delta_f = np.sqrt(np.sum((freqs - f_med)**2 * G_norm2) * df)

    return delta_t, delta_f

# Calcular para diferentes sinais
print(f"{'Sinal':<30} {'Δt':>8} {'Δf':>8} {'Δt·Δf':>10} {'Δt·Δf/(1/4π)':>14}")
print("-" * 74)

resultados = []

# Gaussianas com diferentes larguras
for sigma in [0.3, 0.5, 1.0, 2.0]:
    g = np.exp(-t**2 / (2 * sigma**2))
    dt_rms, df_rms = calcular_delta_t_f(g, t, freqs, dt, df)
    produto = dt_rms * df_rms
    razao = produto / (1 / (4 * np.pi))
    resultados.append((f'Gaussiana (σ={sigma})', dt_rms, df_rms, produto, razao))
    print(f"{'Gaussiana (σ=' + str(sigma) + ')':<30} {dt_rms:>8.4f} {df_rms:>8.4f} {produto:>10.4f} {razao:>14.4f}")

# Pulso retangular com diferentes larguras
for tau in [0.5, 1.0, 2.0]:
    g = np.where(np.abs(t) <= tau/2, 1.0, 0.0)
    dt_rms, df_rms = calcular_delta_t_f(g, t, freqs, dt, df)
    produto = dt_rms * df_rms
    razao = produto / (1 / (4 * np.pi))
    resultados.append((f'Retangular (τ={tau})', dt_rms, df_rms, produto, razao))
    print(f"{'Retangular (τ=' + str(tau) + ')':<30} {dt_rms:>8.4f} {df_rms:>8.4f} {produto:>10.4f} {razao:>14.4f}")

print(f"\nValor mínimo teórico (Gaussiana): Δt·Δf = 1/(4π) ≈ {1/(4*np.pi):.4f}")
print("A Gaussiana atinge (ou se aproxima do) limite inferior — é a função ótima!")

**Observações da Seção 3.7:**

- A DEE $\Psi_g(f) = |G(f)|^2$ mostra como a energia se distribui nas frequências.
- A **largura de banda essencial** é um conceito prático: a maioria da energia está concentrada em uma faixa finita de frequências, mesmo que o espectro se estenda ao infinito.
- O **princípio da incerteza** impõe um limite fundamental: $\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$. A Gaussiana é a única função que atinge esse limite mínimo.

---

<a id="sec3_8"></a>
## Seção 3.8: Densidade Espectral de Potência

Para **sinais de potência** (energia infinita, mas potência média finita), como sinais periódicos e processos aleatórios estacionários, usamos a **Densidade Espectral de Potência** (DEP ou PSD):

$$S_g(f) = \lim_{T \to \infty} \frac{|G_T(f)|^2}{T}$$

onde $G_T(f)$ é a TF do sinal truncado em $[-T/2, T/2]$.

A potência média é:

$$P_g = \int_{-\infty}^{\infty} S_g(f) \, df$$

Para sinais periódicos, a PSD consiste em deltas nas frequências harmônicas.

In [ ]:
# --- 3.8a: PSD de um sinal periódico (onda quadrada) ---

N = 8192
fs = 1000
dt = 1 / fs
t = np.arange(N) * dt
freqs = fftshift(fftfreq(N, dt))

# Onda quadrada (10 Hz)
f0 = 10
x = signal.square(2 * np.pi * f0 * t)

# PSD via periodograma
f_psd, Pxx = signal.periodogram(x, fs, scaling='density')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t, x, 'b', linewidth=1)
axes[0].set_title('Onda quadrada (sinal periódico)')
axes[0].set_xlabel('$t$ (s)'); axes[0].set_xlim(0, 0.5); axes[0].grid(True)

axes[1].semilogy(f_psd, Pxx, 'r', linewidth=1)
axes[1].set_title('Densidade Espectral de Potência (PSD)')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('PSD (V²/Hz)')
axes[1].set_xlim(0, 200); axes[1].grid(True)

# Marcar harmônicas ímpares
for k in range(1, 20, 2):
    axes[1].axvline(x=k * f0, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()
print(f"A onda quadrada possui harmônicas apenas nas frequências ímpares: {f0}, {3*f0}, {5*f0}, ...")

In [ ]:
# --- 3.8b: Senóide com ruído branco — PSD e filtragem ---

N = 16384
fs = 1000
dt = 1 / fs
t = np.arange(N) * dt

# Senóide de 50 Hz + ruído branco
f_sig = 50
A_sig = 1.0
np.random.seed(42)
ruido = np.random.randn(N) * 0.8
x_limpo = A_sig * np.sin(2 * np.pi * f_sig * t)
x_ruidoso = x_limpo + ruido

# PSD via método de Welch (mais suave que periodograma)
f_psd, Pxx_ruidoso = signal.welch(x_ruidoso, fs, nperseg=1024)
f_psd, Pxx_limpo = signal.welch(x_limpo, fs, nperseg=1024)

# Filtrar o sinal ruidoso (Butterworth passa-baixas, fc = 80 Hz)
fc = 80
b, a = signal.butter(6, fc, fs=fs)
x_filtrado = signal.filtfilt(b, a, x_ruidoso)
f_psd, Pxx_filtrado = signal.welch(x_filtrado, fs, nperseg=1024)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Sinais no tempo
axes[0, 0].plot(t, x_ruidoso, 'b', alpha=0.5, linewidth=0.5, label='Ruidoso')
axes[0, 0].plot(t, x_limpo, 'r', linewidth=1.5, label='Original')
axes[0, 0].set_title('Sinais no tempo')
axes[0, 0].set_xlabel('$t$ (s)'); axes[0, 0].set_xlim(0, 0.2)
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(t, x_ruidoso, 'b', alpha=0.5, linewidth=0.5, label='Ruidoso')
axes[0, 1].plot(t, x_filtrado, 'g', linewidth=1.5, label='Filtrado')
axes[0, 1].set_title('Resultado da filtragem')
axes[0, 1].set_xlabel('$t$ (s)'); axes[0, 1].set_xlim(0, 0.2)
axes[0, 1].legend(); axes[0, 1].grid(True)

# PSD
axes[1, 0].semilogy(f_psd, Pxx_ruidoso, 'b', alpha=0.7, label='Ruidoso')
axes[1, 0].semilogy(f_psd, Pxx_limpo, 'r', linewidth=2, label='Sinal puro')
axes[1, 0].set_title('PSD: sinal ruidoso vs puro')
axes[1, 0].set_xlabel('$f$ (Hz)'); axes[1, 0].set_ylabel('PSD (V²/Hz)')
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].semilogy(f_psd, Pxx_ruidoso, 'b', alpha=0.5, label='Antes do filtro')
axes[1, 1].semilogy(f_psd, Pxx_filtrado, 'g', linewidth=2, label='Após filtro')
axes[1, 1].axvline(x=fc, color='gray', linestyle='--', label=f'$f_c = {fc}$ Hz')
axes[1, 1].set_title('PSD: antes e depois da filtragem')
axes[1, 1].set_xlabel('$f$ (Hz)'); axes[1, 1].set_ylabel('PSD (V²/Hz)')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Calcular SNR antes e depois
P_sinal = np.mean(x_limpo**2)
P_ruido_antes = np.mean((x_ruidoso - x_limpo)**2)
P_ruido_depois = np.mean((x_filtrado - x_limpo)**2)
SNR_antes = 10 * np.log10(P_sinal / P_ruido_antes)
SNR_depois = 10 * np.log10(P_sinal / P_ruido_depois)

print(f"SNR antes da filtragem:  {SNR_antes:.1f} dB")
print(f"SNR depois da filtragem: {SNR_depois:.1f} dB")
print(f"Melhoria de SNR: {SNR_depois - SNR_antes:.1f} dB")

In [ ]:
# --- 3.8c: Exemplo prático — cálculo de SNR em enlace de comunicação ---

print("=" * 60)
print("EXEMPLO PRÁTICO: Cálculo de SNR em enlace de comunicação")
print("=" * 60)

# Parâmetros do enlace
P_tx_dBm = 20         # Potência transmitida (dBm)
L_cabo_dB = 10        # Perda no cabo (dB)
G_ant_dB = 5           # Ganho da antena (dB)
L_prop_dB = 80         # Perda de propagação (dB)
N0_dBm_Hz = -174       # Densidade de ruído térmico a 290 K (dBm/Hz)
BW_Hz = 200e3          # Largura de banda do receptor (Hz)

# Cálculos
P_rx_dBm = P_tx_dBm - L_cabo_dB + G_ant_dB - L_prop_dB + G_ant_dB
N_dBm = N0_dBm_Hz + 10 * np.log10(BW_Hz)
SNR_dB = P_rx_dBm - N_dBm

print(f"\nPotência transmitida:        {P_tx_dBm:>8.1f} dBm")
print(f"Perda no cabo:              -{L_cabo_dB:>8.1f} dB")
print(f"Ganho da antena (Tx):       +{G_ant_dB:>8.1f} dB")
print(f"Perda de propagação:        -{L_prop_dB:>8.1f} dB")
print(f"Ganho da antena (Rx):       +{G_ant_dB:>8.1f} dB")
print(f"{'─' * 45}")
print(f"Potência recebida:           {P_rx_dBm:>8.1f} dBm")
print(f"\nDensidade de ruído (N₀):     {N0_dBm_Hz:>8.1f} dBm/Hz")
print(f"Largura de banda:            {BW_Hz/1e3:>8.1f} kHz")
print(f"Potência de ruído:           {N_dBm:>8.1f} dBm")
print(f"{'─' * 45}")
print(f"SNR no receptor:             {SNR_dB:>8.1f} dB")

# Verificar se SNR é suficiente para QPSK (requer ~10 dB para BER < 10⁻⁵)
limiar_SNR = 10
if SNR_dB > limiar_SNR:
    print(f"\n✓ SNR = {SNR_dB:.1f} dB > {limiar_SNR} dB → Enlace viável para QPSK!")
else:
    print(f"\n✗ SNR = {SNR_dB:.1f} dB < {limiar_SNR} dB → Enlace insuficiente para QPSK.")

**Observações da Seção 3.8:**

- A **PSD** é a ferramenta adequada para sinais de potência (periódicos e aleatórios). O método de **Welch** (média de periodogramas com overlap) fornece estimativas mais suaves.
- A filtragem reduz a potência de ruído ao limitar a largura de banda, melhorando a SNR.
- Em enlaces de comunicação, o cálculo de SNR envolve o balanço entre potência do sinal e potência de ruído na largura de banda do receptor.

---

<a id="sec3_9"></a>
## Seção 3.9: Transformada Discreta de Fourier (DFT) e FFT

A **DFT** é a versão discreta e finita da TF, aplicável a sinais amostrados:

$$X[k] = \sum_{n=0}^{N-1} x[n] \, e^{-j2\pi kn/N}, \quad k = 0, 1, \ldots, N-1$$

A **FFT** (Fast Fourier Transform) é um algoritmo eficiente para calcular a DFT:
- DFT direta: $O(N^2)$ operações
- FFT (Cooley-Tukey): $O(N \log N)$ operações

Nesta seção, exploraremos:
- Implementação manual da DFT vs FFT
- Aliasing (subamostragem)
- Vazamento espectral (leakage) e janelamento
- Zero-padding
- Exemplos práticos: análise espectral e espectrograma

In [ ]:
# --- 3.9a: Implementação manual da DFT vs numpy.fft.fft ---
import time

def dft_manual(x):
    """Implementação direta da DFT — O(N²)."""
    N = len(x)
    X = np.zeros(N, dtype=complex)
    n = np.arange(N)
    for k in range(N):
        X[k] = np.sum(x * np.exp(-1j * 2 * np.pi * k * n / N))
    return X

# Testar com sinais de diferentes tamanhos
tamanhos = [64, 128, 256, 512, 1024]
tempos_dft = []
tempos_fft = []

# Sinal de teste
for N in tamanhos:
    x = np.random.randn(N)

    t0 = time.time()
    X_dft = dft_manual(x)
    t_dft = time.time() - t0
    tempos_dft.append(t_dft)

    t0 = time.time()
    X_fft = npfft(x)
    t_fft = time.time() - t0
    tempos_fft.append(t_fft)

# Verificar equivalência (para o menor tamanho)
x_test = np.random.randn(64)
X_dft_test = dft_manual(x_test)
X_fft_test = npfft(x_test)
erro = np.max(np.abs(X_dft_test - X_fft_test))
print(f"Erro máximo entre DFT manual e FFT (N=64): {erro:.2e}")

# Comparar tempos
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(tamanhos, tempos_dft, 'ro-', linewidth=2, markersize=8, label='DFT manual $O(N^2)$')
ax.semilogy(tamanhos, tempos_fft, 'bs-', linewidth=2, markersize=8, label='FFT $O(N \\log N)$')
ax.set_title('Comparação de tempo: DFT manual vs FFT')
ax.set_xlabel('Tamanho N'); ax.set_ylabel('Tempo (s)')
ax.legend(); ax.grid(True)

# Adicionar razão
for i, N in enumerate(tamanhos):
    if tempos_fft[i] > 0:
        razao = tempos_dft[i] / max(tempos_fft[i], 1e-10)
        ax.annotate(f'{razao:.0f}x', (N, tempos_dft[i]), textcoords="offset points",
                    xytext=(10, 0), fontsize=9)

plt.tight_layout()
plt.show()
print(f"\nPara N={tamanhos[-1]}: DFT = {tempos_dft[-1]:.4f}s, FFT = {tempos_fft[-1]:.6f}s")

In [ ]:
# --- 3.9b: Aliasing — efeito da subamostragem ---
# Teorema de Nyquist: fs >= 2·fmax para evitar aliasing

f_sinal = 50  # Hz — frequência do sinal
t_cont = np.linspace(0, 0.1, 10000)  # "contínuo" (muitas amostras)
x_cont = np.sin(2 * np.pi * f_sinal * t_cont)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

taxas = [200, 120, 80, 40]  # Taxas de amostragem (Hz)
titles = [
    f'$f_s = {200}$ Hz > $2f_{{max}}$ (OK)',
    f'$f_s = {120}$ Hz > $2f_{{max}}$ (OK, mas justo)',
    f'$f_s = {80}$ Hz < $2f_{{max}}$ (aliasing!)',
    f'$f_s = {40}$ Hz < $2f_{{max}}$ (aliasing severo!)'
]

for i, (fs_i, titulo) in enumerate(zip(taxas, titles)):
    ax = axes[i // 2, i % 2]
    t_disc = np.arange(0, 0.1, 1/fs_i)
    x_disc = np.sin(2 * np.pi * f_sinal * t_disc)

    ax.plot(t_cont, x_cont, 'b-', alpha=0.3, linewidth=1, label='Sinal original (50 Hz)')
    ax.stem(t_disc, x_disc, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Amostras ($f_s={fs_i}$)')

    # Se houver aliasing, mostrar a frequência aliased
    if fs_i < 2 * f_sinal:
        f_alias = np.abs(f_sinal - fs_i * round(f_sinal / fs_i))
        x_alias = np.sin(2 * np.pi * f_alias * t_cont)
        ax.plot(t_cont, x_alias, 'g--', linewidth=2, label=f'Alias: {f_alias} Hz')

    ax.set_title(titulo)
    ax.set_xlabel('$t$ (s)'); ax.set_xlim(0, 0.1)
    ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout()
plt.show()
print("Quando fs < 2·fmax, o sinal amostrado não pode ser distinguido de um sinal")
print("de frequência mais baixa (alias). Esse é o fenômeno de ALIASING.")

In [ ]:
# --- 3.9c: Vazamento espectral (spectral leakage) e janelamento ---

N = 256
fs = 256  # resolução = fs/N = 1 Hz
t = np.arange(N) / fs

# Caso 1: frequência cai exatamente em um bin (sem leakage)
f_exato = 20.0  # Hz — cai no bin 20
x_exato = np.sin(2 * np.pi * f_exato * t)

# Caso 2: frequência NÃO cai em um bin (com leakage)
f_leak = 20.5  # Hz — entre bins 20 e 21
x_leak = np.sin(2 * np.pi * f_leak * t)

# Aplicar janela de Hanning ao caso com leakage
janela = np.hanning(N)
x_leak_jan = x_leak * janela

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
freqs = fftfreq(N, 1/fs)[:N//2]

# Sem leakage
X_exato = np.abs(npfft(x_exato))[:N//2] * 2 / N
axes[0].plot(freqs, X_exato, 'b', linewidth=2)
axes[0].set_title(f'Sem leakage: $f = {f_exato}$ Hz\n(cai exatamente no bin)')
axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_ylabel('Amplitude')
axes[0].set_xlim(0, 50); axes[0].grid(True)

# Com leakage (sem janela)
X_leak = np.abs(npfft(x_leak))[:N//2] * 2 / N
axes[1].plot(freqs, X_leak, 'r', linewidth=2)
axes[1].set_title(f'Com leakage: $f = {f_leak}$ Hz\n(entre bins — janela retangular)')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, 50); axes[1].grid(True)

# Com janela de Hanning
X_leak_jan = np.abs(npfft(x_leak_jan))[:N//2] * 2 / N
# Compensar perda de amplitude da janela
X_leak_jan *= 2  # fator de correção da Hanning
axes[2].plot(freqs, X_leak_jan, 'g', linewidth=2)
axes[2].set_title(f'Janela de Hanning: $f = {f_leak}$ Hz\n(leakage reduzido)')
axes[2].set_xlabel('$f$ (Hz)'); axes[2].set_ylabel('Amplitude')
axes[2].set_xlim(0, 50); axes[2].grid(True)

plt.tight_layout()
plt.show()
print("A janela de Hanning reduz os lóbulos laterais (leakage), mas alarga o lóbulo principal.")

In [ ]:
# --- 3.9d: Zero-padding — melhor resolução visual (interpolação espectral) ---

N = 64
fs = 64
t = np.arange(N) / fs

# Sinal com duas frequências próximas
f1, f2 = 10, 13
x = np.sin(2 * np.pi * f1 * t) + 0.7 * np.sin(2 * np.pi * f2 * t)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

paddings = [0, 4, 16]  # Fator de zero-padding
titles = ['Sem zero-padding (N=64)', 'Zero-padding 4x (N=256)', 'Zero-padding 16x (N=1024)']

for i, (pad_factor, titulo) in enumerate(zip(paddings, titles)):
    N_pad = N * (pad_factor if pad_factor > 0 else 1)
    x_pad = np.zeros(N_pad)
    x_pad[:N] = x

    freqs_pad = fftfreq(N_pad, 1/fs)[:N_pad//2]
    X_pad = np.abs(npfft(x_pad))[:N_pad//2] * 2 / N  # normalizar por N original

    axes[i].plot(freqs_pad, X_pad, 'b-o' if pad_factor == 0 else 'b-',
                 linewidth=2, markersize=4 if pad_factor == 0 else 0)
    axes[i].set_title(titulo)
    axes[i].set_xlabel('$f$ (Hz)'); axes[i].set_ylabel('Amplitude')
    axes[i].set_xlim(0, 30); axes[i].grid(True)
    axes[i].axvline(x=f1, color='r', linestyle='--', alpha=0.5, label=f'{f1} Hz')
    axes[i].axvline(x=f2, color='g', linestyle='--', alpha=0.5, label=f'{f2} Hz')
    axes[i].legend()

plt.tight_layout()
plt.show()
print("Zero-padding NÃO aumenta a resolução espectral real (que depende da duração do sinal),")
print("mas interpola entre os bins, revelando melhor a forma do espectro.")

In [ ]:
# --- 3.9e: Análise espectral de sinal multi-tom (simulando áudio) ---

N = 4096
fs = 8000  # Taxa de amostragem típica de telefonia
t = np.arange(N) / fs

# Simular tons DTMF (Dual-Tone Multi-Frequency) — tecla "5"
f_low = 770    # Hz (frequência baixa da linha 2)
f_high = 1336  # Hz (frequência alta da coluna 2)
x = np.sin(2 * np.pi * f_low * t) + np.sin(2 * np.pi * f_high * t)

# Adicionar um pouco de ruído
np.random.seed(42)
x_noisy = x + 0.1 * np.random.randn(N)

# Espectro
freqs_fft = fftfreq(N, 1/fs)[:N//2]
X = np.abs(npfft(x_noisy))[:N//2] * 2 / N

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t * 1000, x_noisy, 'b', linewidth=0.5)
axes[0].set_title('Sinal DTMF "5" no tempo')
axes[0].set_xlabel('$t$ (ms)'); axes[0].set_xlim(0, 20); axes[0].grid(True)

axes[1].plot(freqs_fft, X, 'r', linewidth=1)
axes[1].axvline(x=f_low, color='blue', linestyle='--', alpha=0.7, label=f'{f_low} Hz')
axes[1].axvline(x=f_high, color='green', linestyle='--', alpha=0.7, label=f'{f_high} Hz')
axes[1].set_title('Espectro: tons DTMF identificados')
axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, 2000)
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f"Tons DTMF detectados: {f_low} Hz e {f_high} Hz → Tecla '5'")

In [ ]:
# --- 3.9f: Espectrograma de um sinal chirp (frequência variável no tempo) ---

fs = 4000
T = 2.0  # duração em segundos
t = np.arange(0, T, 1/fs)

# Chirp: frequência varia linearmente de 100 Hz a 1500 Hz
f_start = 100
f_end = 1500
x_chirp = signal.chirp(t, f_start, T, f_end, method='linear')

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Sinal no tempo
axes[0].plot(t, x_chirp, 'b', linewidth=0.5)
axes[0].set_title(f'Sinal chirp: frequência de {f_start} Hz a {f_end} Hz')
axes[0].set_xlabel('$t$ (s)'); axes[0].set_ylabel('Amplitude'); axes[0].grid(True)

# Espectrograma
axes[1].specgram(x_chirp, NFFT=256, Fs=fs, noverlap=200, cmap='inferno')
axes[1].set_title('Espectrograma (STFT)')
axes[1].set_xlabel('Tempo (s)'); axes[1].set_ylabel('Frequência (Hz)')
axes[1].set_ylim(0, 2000)

plt.tight_layout()
plt.show()
print("O espectrograma mostra como a frequência instantânea do chirp")
print("varia linearmente com o tempo — é uma representação tempo-frequência.")

**Observações da Seção 3.9:**

- A **DFT manual** tem complexidade $O(N^2)$, enquanto a **FFT** (Cooley-Tukey) tem $O(N \log N)$ — diferença crucial para sinais longos.
- **Aliasing** ocorre quando $f_s < 2 f_{max}$ — frequências altas "dobram" para frequências baixas.
- **Vazamento espectral** ocorre quando a frequência do sinal não cai exatamente em um bin da DFT. Janelas (Hanning, Hamming, Blackman) reduzem o leakage ao custo de alargar o lóbulo principal.
- **Zero-padding** não melhora a resolução espectral real, mas interpola o espectro, facilitando a visualização.
- O **espectrograma** (STFT) é a ferramenta para sinais cuja frequência varia no tempo (como chirps, fala, música).

---

## Demonstrações Interativas (ipywidgets)

As células a seguir utilizam **ipywidgets** para criar demonstrações interativas. Para utilizá-las, é necessário ter o pacote `ipywidgets` instalado (`pip install ipywidgets`) e executar o notebook em um ambiente que suporte widgets (JupyterLab, Jupyter Notebook, VS Code).

In [ ]:
# --- Widget 1: Largura do pulso retangular e seu espectro ---
try:
    from ipywidgets import interact, FloatSlider

    @interact(tau=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                               description='Largura τ:', style={'description_width': '80px'}))
    def pulso_interativo(tau):
        t = np.linspace(-4, 4, 2000)
        f = np.linspace(-10, 10, 2000)

        g = np.where(np.abs(t) <= tau / 2, 1.0, 0.0)
        G = tau * np.sinc(f * tau)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].plot(t, g, 'b', linewidth=2)
        axes[0].set_title(f'Pulso retangular, $\\tau = {tau:.1f}$')
        axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$g(t)$')
        axes[0].set_ylim(-0.2, 1.4); axes[0].grid(True)

        axes[1].plot(f, G, 'r', linewidth=2)
        axes[1].set_title(f'$G(f) = {tau:.1f} \\cdot \\mathrm{{sinc}}({tau:.1f} f)$')
        axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_ylabel('$G(f)$')
        axes[1].axhline(y=0, color='k', linewidth=0.5); axes[1].grid(True)

        plt.tight_layout()
        plt.show()
        print(f"Largura do lóbulo principal (primeiro zero): 1/τ = {1/tau:.2f} Hz")

except ImportError:
    print("ipywidgets não instalado. Execute: pip install ipywidgets")

In [ ]:
# --- Widget 2: Ordem do filtro Butterworth ---
try:
    from ipywidgets import interact, IntSlider

    @interact(ordem=IntSlider(value=2, min=1, max=10, step=1,
                               description='Ordem N:', style={'description_width': '80px'}))
    def filtro_interativo(ordem):
        fc = 100
        fs = 1000
        f = np.linspace(0, fs/2, 5000)

        b, a = signal.butter(ordem, fc, fs=fs)
        _, h = signal.freqz(b, a, worN=f, fs=fs)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Filtro ideal
        H_ideal = np.where(f <= fc, 0, -100)
        axes[0].plot(f, H_ideal, 'k--', linewidth=1, alpha=0.5, label='Ideal')
        axes[0].plot(f, 20 * np.log10(np.maximum(np.abs(h), 1e-10)), 'b', linewidth=2,
                     label=f'Butterworth N={ordem}')
        axes[0].set_title(f'Butterworth ordem {ordem} — Magnitude')
        axes[0].set_xlabel('$f$ (Hz)'); axes[0].set_ylabel('$|H(f)|$ (dB)')
        axes[0].set_ylim(-80, 5); axes[0].axvline(x=fc, color='gray', linestyle=':', alpha=0.5)
        axes[0].axhline(y=-3, color='red', linestyle='--', alpha=0.5, label='-3 dB')
        axes[0].legend(); axes[0].grid(True)

        # Resposta ao impulso
        t_imp = np.zeros(500)
        t_imp[0] = 1
        h_imp = signal.lfilter(b, a, t_imp)
        t_ax = np.arange(500) / fs * 1000

        axes[1].plot(t_ax, h_imp, 'r', linewidth=2)
        axes[1].set_title(f'Resposta ao impulso $h[n]$')
        axes[1].set_xlabel('$t$ (ms)'); axes[1].set_xlim(0, 100); axes[1].grid(True)

        plt.tight_layout()
        plt.show()
        print(f"Atenuação na banda de rejeição (2fc): "
              f"{-20*ordem*np.log10(2):.1f} dB/oitava = {-20*ordem:.0f} dB/década")

except ImportError:
    print("ipywidgets não instalado. Execute: pip install ipywidgets")

In [ ]:
# --- Widget 3: Modulação interativa — frequência da portadora ---
try:
    from ipywidgets import interact, FloatSlider

    @interact(f0=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                              description='f₀ (Hz):', style={'description_width': '80px'}))
    def modulacao_interativa(f0):
        N = 8192
        dt = 0.005
        t = np.arange(-N//2, N//2) * dt
        freqs = fftshift(fftfreq(N, dt))

        g = np.exp(-2 * t**2)
        g_mod = g * np.cos(2 * np.pi * f0 * t)

        G = fftshift(npfft(g)) * dt
        G_mod = fftshift(npfft(g_mod)) * dt

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].plot(t, g, 'b', alpha=0.3, linewidth=1, label='Envoltória')
        axes[0].plot(t, g_mod, 'r', linewidth=1.5, label=f'Modulado ($f_0={f0}$ Hz)')
        axes[0].set_title(f'$g(t) \\cos(2\\pi \\cdot {f0} \\cdot t)$')
        axes[0].set_xlabel('$t$'); axes[0].set_xlim(-3, 3)
        axes[0].legend(); axes[0].grid(True)

        axes[1].plot(freqs, np.abs(G), 'b', alpha=0.3, linewidth=1, label='Banda-base')
        axes[1].plot(freqs, np.abs(G_mod), 'r', linewidth=2, label='Modulado')
        axes[1].axvline(x=f0, color='gray', linestyle='--', alpha=0.5)
        axes[1].axvline(x=-f0, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_title(f'Espectro deslocado para $\\pm {f0}$ Hz')
        axes[1].set_xlabel('$f$ (Hz)'); axes[1].set_xlim(-25, 25)
        axes[1].legend(); axes[1].grid(True)

        plt.tight_layout()
        plt.show()

except ImportError:
    print("ipywidgets não instalado. Execute: pip install ipywidgets")

---

## Resumo do Módulo 1 — Capítulo 3

| Seção | Tópico | Conceito-chave |
|-------|--------|----------------|
| 3.1 | Transformada de Fourier | Representação tempo-frequência, relação inversa entre duração e largura de banda |
| 3.2 | Funções úteis | Pares de transformadas fundamentais (rect/sinc, Gaussiana, exponencial) |
| 3.3 | Propriedades da TF | Linearidade, deslocamento, modulação, escala, convolução, Parseval |
| 3.4 | Sistemas LTI | Filtragem como multiplicação na frequência, função de transferência |
| 3.5 | Filtros ideais vs práticos | Butterworth, Chebyshev, Bessel — compromissos de projeto |
| 3.6 | Distorção de sinais | Distorção de amplitude e fase, equalização |
| 3.7 | Densidade espectral de energia | DEE, largura de banda essencial, princípio da incerteza |
| 3.8 | Densidade espectral de potência | PSD, análise de SNR, filtragem de ruído |
| 3.9 | DFT e FFT | Aliasing, leakage, janelamento, zero-padding, espectrograma |